# Bomberman RL -- GPU training notebook (Colab)

Vectorized NumPy Bomberman env + PPO self-play, adapted to run on a Colab GPU.

**Before running anything:** `Runtime -> Change runtime type -> T4 GPU` (or better, if you have Colab Pro).

What changed vs. the CPU version:
- `PPO(..., device="cuda")` in `train_example.py` / `profile_train.py` so the policy, optimizer, and self-play
  opponent snapshots (`bots.FrozenPolicy` deep-copies `model.policy`, so it inherits the device automatically) run
  on GPU.
- `SmallGridCNN` widened (64/128/128 channels, `features_dim=512`) now that compute is cheap on GPU -- on CPU this
  same width would have made things slower, not faster.
- `isolate_torch_costs.py` moves its model/tensors to `device` and calls `torch.cuda.synchronize()` around the
  timed region, since CUDA kernel launches are asynchronous and an unsynchronized wall-clock would only measure
  how fast Python can *queue* work, not how long it actually takes.
- `sim.py`, `bots.py`, `gym_env.py`, `benchmark.py` are unchanged -- the physics/env layer is NumPy-only and was
  never the bottleneck (it was already doing >150k steps/s).


## 1. Confirm GPU runtime is active

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime > Change runtime type > select a GPU, then re-run this cell.")


## 2. Install dependencies

In [ ]:
!pip install -q stable-baselines3 gymnasium tensorboard

## 3. (Optional) Mount Google Drive

Useful for checkpointing `model.save(...)` and TensorBoard logs so they survive a session disconnect.
Skip this cell if you don't need persistence across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/bomberman_rl_checkpoints"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)


## 4. Write project source files

Each cell below writes one file to the Colab runtime's local disk via `%%writefile`, so the rest of the notebook
can `import` them normally, exactly like running the original scripts locally.

### `sim.py` -- vectorized NumPy game physics (unchanged)

In [ ]:
%%writefile sim.py
"""
Vectorized, dependency-free reimplementation of the Heidelberg `bomberman_rl`
game rules (see environment.py / agents.py / items.py / settings.py).

Design goal: step() advances N independent games at once using plain NumPy
array ops (no Python objects, no pygame, no subprocesses, no logging).
That removes essentially all of the original engine's per-step overhead;
running many envs per call is what turns this into real RL throughput.

Rule fidelity notes (deliberate, documented simplifications):
  - Blast propagation, bomb timers, explosion stage machine (dangerous ->
    smoke -> gone), coin reveal-under-crate, and the "own-bomb self-kill vs.
    opponent-kill" scoring are all reproduced exactly, including the
    original's quirk that if an agent stands where two *different* bombs'
    blasts overlap in the same step, both owners are credited a kill (each
    explosion object is evaluated independently in the original code).
  - Agent turn order within a step is a per-environment random permutation,
    exactly like `step_world`, so earlier movers "claim" a tile first.
"""
from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field

WALL, FREE, CRATE = -1, 0, 1

# Actions
UP, DOWN, LEFT, RIGHT, WAIT, BOMB = 0, 1, 2, 3, 4, 5
ACTION_NAMES = ["UP", "DOWN", "LEFT", "RIGHT", "WAIT", "BOMB"]
DX = np.array([0, 0, -1, 1, 0, 0])
DY = np.array([-1, 1, 0, 0, 0, 0])

SCENARIOS = {
    "empty": dict(CRATE_DENSITY=0.0, COIN_COUNT=0),
    "coin-heaven": dict(CRATE_DENSITY=0.0, COIN_COUNT=50),
    "loot-crate": dict(CRATE_DENSITY=0.75, COIN_COUNT=50),
    "classic": dict(CRATE_DENSITY=0.75, COIN_COUNT=9),
}


@dataclass
class Settings:
    cols: int = 17
    rows: int = 17
    max_steps: int = 400
    bomb_timer: int = 4
    bomb_power: int = 3
    explosion_timer: int = 2      # dangerous stage length
    smoke_timer: int = 2          # cosmetic-only stage length
    reward_kill: int = 5
    reward_coin: int = 1
    scenario: str = "classic"
    max_coins: int = field(default=None)  # derived if None

    def __post_init__(self):
        if self.max_coins is None:
            self.max_coins = SCENARIOS[self.scenario]["COIN_COUNT"]


START_POSITIONS = None  # filled in per Settings instance (depends on cols/rows)


def _start_positions(s: Settings):
    return [(1, 1), (1, s.rows - 2), (s.cols - 2, 1), (s.cols - 2, s.rows - 2)]


# Event flags returned per (env, agent) each step, as a bit-packed uint16.
EV_MOVED = 1 << 0
EV_INVALID = 1 << 1
EV_WAITED = 1 << 2
EV_BOMB_DROPPED = 1 << 3
EV_BOMB_EXPLODED = 1 << 4
EV_CRATE_DESTROYED = 1 << 5
EV_COIN_FOUND = 1 << 6
EV_COIN_COLLECTED = 1 << 7
EV_KILLED_SELF = 1 << 8
EV_KILLED_OPPONENT = 1 << 9
EV_GOT_KILLED = 1 << 10
EV_OPPONENT_ELIMINATED = 1 << 11
EV_SURVIVED_ROUND = 1 << 12


class VecBomberman:
    """
    N independent games, stepped together.

    State (all NumPy arrays, leading dim N unless noted):
      arena            (N, C, R) int8         -1 wall, 0 free, 1 crate
      coin_xy           (N, K, 2) int16        K = max_coins, padded with -1
      coin_state        (N, K)   int8          0 hidden, 1 collectable, 2 collected
      agent_xy          (N, 4, 2) int16
      agent_alive        (N, 4)   bool
      agent_score        (N, 4)   int32
      bomb_active         (N, 4)   bool         indexed by *owner* slot (max 1 bomb/agent)
      bomb_xy             (N, 4, 2) int16
      bomb_timer          (N, 4)   int8
      expl_active         (N, 4)   bool
      expl_dangerous       (N, 4)   bool
      expl_timer            (N, 4)   int8
      expl_mask              (N, 4, C, R) bool   fixed at creation
      step_no                 (N,)     int32
      running                   (N,)     bool
    """

    def __init__(self, n_envs: int, settings: Settings | None = None, seed: int = 0):
        self.n = n_envs
        self.s = settings or Settings()
        self.rng = np.random.default_rng(seed)
        C, R = self.s.cols, self.s.rows
        self.C, self.R = C, R
        K = self.s.max_coins
        self.K = K

        self.arena = np.zeros((self.n, C, R), dtype=np.int8)
        self.coin_xy = np.full((self.n, K, 2), -1, dtype=np.int16)
        self.coin_state = np.zeros((self.n, K), dtype=np.int8)
        self.agent_xy = np.zeros((self.n, 4, 2), dtype=np.int16)
        self.agent_alive = np.zeros((self.n, 4), dtype=bool)
        self.agent_score = np.zeros((self.n, 4), dtype=np.int32)
        self.bomb_active = np.zeros((self.n, 4), dtype=bool)
        self.bomb_xy = np.zeros((self.n, 4, 2), dtype=np.int16)
        self.bomb_timer = np.zeros((self.n, 4), dtype=np.int8)
        self.expl_active = np.zeros((self.n, 4), dtype=bool)
        self.expl_dangerous = np.zeros((self.n, 4), dtype=bool)
        self.expl_timer = np.zeros((self.n, 4), dtype=np.int8)
        self.expl_mask = np.zeros((self.n, 4, C, R), dtype=bool)
        self.step_no = np.zeros(self.n, dtype=np.int32)
        self.running = np.zeros(self.n, dtype=bool)

        self.reset_all()

    # ---------------------------------------------------------------- reset
    def reset_all(self):
        idx = np.arange(self.n)
        self.reset(idx)

    def reset(self, env_idx: np.ndarray):
        """Re-initializes the listed environments (build_arena equivalent).
        Not vectorized across envs (uses per-env RNG draws) -- this runs
        once per episode, not per step, so it doesn't need to be fast.
        """
        s = self.s
        C, R = self.C, self.R
        starts = _start_positions(s)
        info = SCENARIOS[s.scenario]

        # Hoisted out of the per-env loop: identical every time, no need to
        # recompute per environment (this was the dominant reset() cost).
        xs, ys = np.meshgrid(np.arange(C), np.arange(R), indexing="ij")
        checker = ((xs + 1) * (ys + 1)) % 2 == 1
        all_pos = np.stack((xs, ys), -1)
        border = np.zeros((C, R), dtype=bool)
        border[:1, :] = border[-1:, :] = border[:, :1] = border[:, -1:] = True
        fixed_walls = checker | border
        clear_positions = set()
        for (x, y) in starts:
            for (xx, yy) in [(x, y), (x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]:
                clear_positions.add((xx, yy))

        for n in env_idx:
            arena = np.zeros((C, R), dtype=np.int8)
            arena[self.rng.random((C, R)) < info["CRATE_DENSITY"]] = CRATE
            arena[fixed_walls] = WALL
            for (xx, yy) in clear_positions:
                if arena[xx, yy] == CRATE:
                    arena[xx, yy] = FREE

            crate_pos = self.rng.permutation(all_pos[arena == CRATE])
            free_pos = self.rng.permutation(all_pos[arena == FREE])
            coin_pos = np.concatenate([crate_pos, free_pos], axis=0)[: s.max_coins]

            self.arena[n] = arena
            self.coin_xy[n] = -1
            self.coin_state[n] = 0
            for k, (x, y) in enumerate(coin_pos):
                self.coin_xy[n, k] = (x, y)
                self.coin_state[n, k] = 1 if arena[x, y] == FREE else 0

            order = self.rng.permutation(4)
            for slot, start_i in enumerate(order):
                x, y = starts[start_i]
                self.agent_xy[n, slot] = (x, y)
            self.agent_alive[n] = True
            self.agent_score[n] = 0
            self.bomb_active[n] = False
            self.bomb_timer[n] = 0
            self.expl_active[n] = False
            self.expl_dangerous[n] = False
            self.expl_timer[n] = 0
            self.expl_mask[n] = False
            self.step_no[n] = 0
            self.running[n] = True

    # ----------------------------------------------------------------- step
    def step(self, actions: np.ndarray):
        """
        actions: (N, 4) int array of action ids (one per agent slot).
        Dead / inactive agents' actions are ignored.

        Returns:
          events   (N, 4) uint16  bit-packed EV_* flags for this step
          done     (N,)   bool    whether the round ended this step
        """
        assert actions.shape == (self.n, 4)
        N, C, R = self.n, self.C, self.R
        events = np.zeros((N, 4), dtype=np.uint16)

        alive = self.agent_alive
        self.step_no += self.running.astype(np.int32)

        # ---- 1. agents act, in a random per-env permutation, sequentially
        perm = np.argsort(self.rng.random((N, 4)), axis=1)  # (N,4) random perm of slots
        occ_bomb = self.bomb_active.copy()  # tile occupancy check uses "any bomb", not owner id

        for t in range(4):
            slot = perm[:, t]                       # (N,) which agent acts this turn
            rows = np.arange(N)
            act = actions[rows, slot]
            active = alive[rows, slot] & self.running
            if not active.any():
                continue

            x = self.agent_xy[rows, slot, 0].astype(np.int32)
            y = self.agent_xy[rows, slot, 1].astype(np.int32)

            for a, dx, dy in ((UP, 0, -1), (DOWN, 0, 1), (LEFT, -1, 0), (RIGHT, 1, 0)):
                sel = active & (act == a)
                if not sel.any():
                    continue
                nx, ny = x + dx, y + dy
                free = self.arena[rows, nx, ny] == FREE
                occ_by_bomb = occ_bomb[rows] & (self.bomb_xy[rows, :, 0] == nx[:, None]) & \
                              (self.bomb_xy[rows, :, 1] == ny[:, None])
                occ_by_bomb = occ_by_bomb.any(axis=1)
                occ_by_agent = alive & (self.agent_xy[..., 0] == nx[:, None]) & \
                               (self.agent_xy[..., 1] == ny[:, None])
                occ_by_agent = occ_by_agent.any(axis=1)
                can_move = sel & free & ~occ_by_bomb & ~occ_by_agent
                self.agent_xy[rows[can_move], slot[can_move], 0] = nx[can_move]
                self.agent_xy[rows[can_move], slot[can_move], 1] = ny[can_move]
                events[rows[can_move], slot[can_move]] |= EV_MOVED
                invalid = sel & ~can_move
                events[rows[invalid], slot[invalid]] |= EV_INVALID

            sel = active & (act == WAIT)
            events[rows[sel], slot[sel]] |= EV_WAITED

            sel = active & (act == BOMB) & (~self.bomb_active[rows, slot])
            if sel.any():
                self.bomb_active[rows[sel], slot[sel]] = True
                self.bomb_xy[rows[sel], slot[sel], 0] = x[sel]
                self.bomb_xy[rows[sel], slot[sel], 1] = y[sel]
                self.bomb_timer[rows[sel], slot[sel]] = self.s.bomb_timer
                events[rows[sel], slot[sel]] |= EV_BOMB_DROPPED
                occ_bomb[rows[sel], slot[sel]] = True
            invalid_bomb = active & (act == BOMB) & self.bomb_active[rows, slot]
            events[rows[invalid_bomb], slot[invalid_bomb]] |= EV_INVALID

        # ---- 2. collect coins
        for k in range(self.K):
            collectable = self.coin_state[:, k] == 1
            if not collectable.any():
                continue
            cx = self.coin_xy[:, k, 0]
            cy = self.coin_xy[:, k, 1]
            hit = alive & (self.agent_xy[..., 0] == cx[:, None]) & (self.agent_xy[..., 1] == cy[:, None])
            hit &= collectable[:, None]
            got = hit.any(axis=1)
            if got.any():
                self.coin_state[got, k] = 2
                first = np.argmax(hit[got], axis=1)  # first agent (by slot order) at that tile
                envs = np.nonzero(got)[0]
                self.agent_score[envs, first] += self.s.reward_coin
                events[envs, first] |= EV_COIN_COLLECTED

        # ---- 3. progress explosions (stage machine) -- BEFORE bombs, as in original
        dec = self.expl_active & self.running[:, None]
        self.expl_timer[dec] -= 1
        flip = dec & (self.expl_timer <= 0) & self.expl_dangerous
        # dangerous -> smoke: bomb owner regains ability to place a bomb
        self.expl_dangerous[flip] = False
        self.expl_timer[flip] = self.s.smoke_timer
        expire = dec & (self.expl_timer <= 0) & ~self.expl_dangerous
        self.expl_active[expire] = False
        self.expl_mask[expire] = False

        # ---- 4. progress bombs / explode
        due = self.bomb_active & self.running[:, None] & (self.bomb_timer <= 0)
        if due.any():
            env_ids, slots = np.nonzero(due)
            for n, slot in zip(env_ids, slots):
                bx, by = int(self.bomb_xy[n, slot, 0]), int(self.bomb_xy[n, slot, 1])
                mask = np.zeros((C, R), dtype=bool)
                mask[bx, by] = True
                for ddx, ddy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    x, y = bx, by
                    for _ in range(self.s.bomb_power):
                        x, y = x + ddx, y + ddy
                        if self.arena[n, x, y] == WALL:
                            break
                        mask[x, y] = True
                # destroy crates, reveal coins
                crate_hit = mask & (self.arena[n] == CRATE)
                if crate_hit.any():
                    self.arena[n][crate_hit] = FREE
                    events[n, slot] |= EV_CRATE_DESTROYED
                    for k in range(self.K):
                        if self.coin_state[n, k] == 0:
                            cx, cy = self.coin_xy[n, k]
                            if cx >= 0 and crate_hit[cx, cy]:
                                self.coin_state[n, k] = 1
                                events[n, slot] |= EV_COIN_FOUND
                events[n, slot] |= EV_BOMB_EXPLODED
                self.bomb_active[n, slot] = False
                self.expl_active[n, slot] = True
                self.expl_dangerous[n, slot] = True
                self.expl_timer[n, slot] = self.s.explosion_timer
                self.expl_mask[n, slot] = mask
        not_due = self.bomb_active & self.running[:, None] & ~due
        self.bomb_timer[not_due] -= 1

        # ---- 5. evaluate explosions: kill agents (mirrors per-explosion-object
        #      independent evaluation, incl. the multi-credit overlap quirk)
        got_hit = np.zeros((N, 4), dtype=bool)
        for owner_slot in range(4):
            dangerous = self.expl_dangerous[:, owner_slot] & self.running
            if not dangerous.any():
                continue
            mask = self.expl_mask[:, owner_slot, :, :]
            for i in range(4):
                a_alive = alive[:, i] & dangerous
                if not a_alive.any():
                    continue
                ax = self.agent_xy[:, i, 0]
                ay = self.agent_xy[:, i, 1]
                hit = a_alive & mask[np.arange(N), ax, ay]
                if not hit.any():
                    continue
                got_hit[hit, i] = True
                self_kill = hit & (owner_slot == i)
                opp_kill = hit & ~self_kill
                events[self_kill, i] |= EV_KILLED_SELF
                if opp_kill.any():
                    self.agent_score[opp_kill, owner_slot] += self.s.reward_kill
                    events[opp_kill, owner_slot] |= EV_KILLED_OPPONENT

        if got_hit.any():
            self.agent_alive[got_hit] = False
            events[got_hit.any(axis=1)] |= 0  # no-op, kept for symmetry
            envs_with_kill = np.nonzero(got_hit.any(axis=1))[0]
            for n in envs_with_kill:
                dead_here = np.nonzero(got_hit[n])[0]
                for d in dead_here:
                    events[n, d] |= EV_GOT_KILLED
                still_alive = np.nonzero(self.agent_alive[n])[0]
                for a in still_alive:
                    events[n, a] |= EV_OPPONENT_ELIMINATED

        # ---- 6. termination check (time_to_stop equivalent)
        n_alive = self.agent_alive.sum(axis=1)
        crates_left = (self.arena == CRATE).any(axis=(1, 2))
        coins_left = (self.coin_state == 1).any(axis=1)
        bombs_or_expl = self.bomb_active.any(axis=1) | self.expl_active.any(axis=1)
        stalemate = (n_alive == 1) & ~crates_left & ~coins_left & ~bombs_or_expl
        timeout = self.step_no >= self.s.max_steps
        done = self.running & ((n_alive == 0) | stalemate | timeout)

        if done.any():
            survivors = self.agent_alive & done[:, None]
            events[survivors] |= EV_SURVIVED_ROUND
            self.running[done] = False

        return events, done

    # ------------------------------------------------------------- helpers
    def auto_reset_done(self, done: np.ndarray):
        """Call after step() to immediately restart any finished envs
        (standard for high-throughput vectorized RL training)."""
        idx = np.nonzero(done)[0]
        if len(idx):
            self.reset(idx)

    def dangerous_mask(self) -> np.ndarray:
        """(N, C, R) bool -- tiles covered by a currently-lethal explosion."""
        out = np.zeros((self.n, self.C, self.R), dtype=bool)
        for slot in range(4):
            dangerous = self.expl_dangerous[:, slot]
            if not dangerous.any():
                continue
            out |= self.expl_mask[:, slot, :, :] & dangerous[:, None, None]
        return out

    def danger_map(self) -> np.ndarray:
        """(N, C, R) countdown grid matching the original's explosion_map
        (max over dangerous explosions covering each tile, value = timer-1)."""
        out = np.zeros((self.n, self.C, self.R), dtype=np.int8)
        for slot in range(4):
            dangerous = self.expl_dangerous[:, slot]
            if not dangerous.any():
                continue
            val = np.maximum(self.expl_timer[:, slot] - 1, 0)
            contrib = self.expl_mask[:, slot, :, :] & dangerous[:, None, None]
            out = np.where(contrib, np.maximum(out, val[:, None, None]), out)
        return out


### `bots.py` -- opponent policies incl. self-play pool (unchanged -- inherits GPU device automatically)

In [ ]:
%%writefile bots.py
"""Cheap vectorized opponent policies, used to fill in non-learning agent
slots. All operate on the whole batch at once (no per-env Python loop) so
they don't become the bottleneck at high num_envs."""
import numpy as np
from sim import VecBomberman, FREE, UP, DOWN, LEFT, RIGHT, WAIT, BOMB


class RandomBot:
    def __call__(self, sim: VecBomberman, slots: np.ndarray) -> np.ndarray:
        return np.random.randint(0, 6, size=slots.shape)


class FrozenPolicy:
    """Cheap, read-only snapshot of an SB3 policy network for use as a
    self-play opponent. Deep-copies just the policy (not the whole PPO
    model/optimizer/rollout buffer), so taking a snapshot every N steps
    stays lightweight compared to model.save()/load() round trips."""

    def __init__(self, policy):
        import copy
        self.policy = copy.deepcopy(policy)
        self.policy.set_training_mode(False)

    def predict(self, obs, deterministic: bool = False):
        import torch
        with torch.no_grad():
            obs_t, _ = self.policy.obs_to_tensor(obs)
            actions, _, _ = self.policy(obs_t, deterministic=deterministic)
        return actions.cpu().numpy(), None


class SelfPlayOpponent:
    """Fills the non-learning slots using a rotating pool of frozen
    snapshots of the learning agent's own policy (see FrozenPolicy).

    Each env's opponent slots are assigned a uniformly random snapshot
    from the pool per call, so the learner faces a mix of recent and
    older versions of itself rather than only its latest (strongest)
    self -- this is the usual guard against self-play strategy collapse /
    cycling. Before any snapshot exists, falls back to `fallback` (a
    plain random policy by default) so training can start immediately.

    Call `add_snapshot(model.policy)` periodically (e.g. from an SB3
    training callback) to keep the pool up to date; oldest snapshots are
    dropped once `pool_size` is exceeded.
    """

    def __init__(self, pool_size: int = 5, fallback=None):
        from gym_env import encode_obs  # local import: avoids bots<->gym_env cycle
        self._encode_obs = encode_obs
        self.pool_size = pool_size
        self.policies: list[FrozenPolicy] = []
        self.fallback = fallback or RandomBot()

    def add_snapshot(self, policy):
        self.policies.append(FrozenPolicy(policy))
        if len(self.policies) > self.pool_size:
            self.policies.pop(0)

    def __call__(self, sim: VecBomberman, slots: np.ndarray) -> np.ndarray:
        N, k = slots.shape
        if not self.policies:
            return self.fallback(sim, slots)

        # Same pool-index-per-env used for every slot (as before), but
        # inference is grouped per policy across ALL slots in one call
        # each, instead of one call per (slot, policy) pair. With k=3
        # slots and a 5-member pool that's <=5 forward passes per env
        # step instead of up to 15 -- each processing k times more rows,
        # so the fixed per-call Python/torch overhead (tensor creation,
        # autograd-context entry, etc.) is paid a third as often. This
        # doesn't change the total amount of inference *compute*, but
        # that fixed overhead -- not FLOPs -- is what dominates at small
        # per-call batch sizes, which is why this alone can be a large
        # wall-clock win.
        actions = np.zeros((N, k), dtype=np.int64)
        pool_idx = np.random.randint(0, len(self.policies), size=N)
        obs_per_slot = [self._encode_obs(sim, slots[:, j]) for j in range(k)]

        for p_i, policy in enumerate(self.policies):
            sel = pool_idx == p_i
            if not sel.any():
                continue
            # Stack this policy's assigned envs across all k slots into
            # one batch: shape (len(sel_envs) * k, channels, C, R).
            batched_obs = np.concatenate([obs_per_slot[j][sel] for j in range(k)], axis=0)
            act, _ = policy.predict(batched_obs, deterministic=False)
            act = act.reshape(k, -1).T  # back to (n_sel_envs, k)
            actions[sel] = act
        return actions


class RandomSafeBot:
    """Like RandomBot, but never walks into a currently-dangerous tile and
    never re-drops a bomb it can't. Cheap heuristic, much less suicidal than
    pure random -- makes for a more useful training partner and for
    benchmarks that reflect realistic (non-degenerate) episode lengths."""

    def __call__(self, sim: VecBomberman, slots: np.ndarray) -> np.ndarray:
        N, k = slots.shape
        danger = sim.dangerous_mask()  # (N,C,R) bool -- tiles currently lethal
        actions = np.random.randint(0, 6, size=(N, k))
        rows = np.arange(N)
        for j in range(k):
            slot = slots[:, j]
            x = sim.agent_xy[rows, slot, 0].astype(np.int32)
            y = sim.agent_xy[rows, slot, 1].astype(np.int32)
            for a, dx, dy in ((UP, 0, -1), (DOWN, 0, 1), (LEFT, -1, 0), (RIGHT, 1, 0)):
                sel = actions[:, j] == a
                if not sel.any():
                    continue
                nx, ny = np.clip(x + dx, 0, sim.C - 1), np.clip(y + dy, 0, sim.R - 1)
                bad = sel & ((sim.arena[rows, nx, ny] != FREE) | danger[rows, nx, ny])
                actions[bad, j] = WAIT
            drop = actions[:, j] == BOMB
            no_bomb = drop & sim.bomb_active[rows, slot]
            actions[no_bomb, j] = WAIT
        return actions

### `gym_env.py` -- Gym/SB3 VecEnv wrappers + observation encoding (unchanged)

In [ ]:
%%writefile gym_env.py
"""
Gym-facing wrappers around sim.VecBomberman.

Two entry points:
  BombermanEnv     - standard gymnasium.Env (single game, single learning
                      agent in slot 0; the other 3 slots are driven by a
                      pluggable opponent policy). Use this for algorithms
                      that expect the classic single-env Gym API.

  BombermanVecEnv   - stable-baselines3 VecEnv. This is the one that gives
                      you the real speed: it steps `num_envs` games per
                      call using sim.VecBomberman directly (no per-env
                      Python loop, no subprocesses). Point PPO/A2C/etc at
                      this with num_envs in the hundreds/thousands.
"""
from __future__ import annotations

import numpy as np
import gymnasium as gym
from gymnasium import spaces

from sim import (
    VecBomberman, Settings, ACTION_NAMES, WALL, CRATE,
    EV_COIN_COLLECTED, EV_KILLED_OPPONENT, EV_KILLED_SELF, EV_GOT_KILLED,
    EV_INVALID, EV_CRATE_DESTROYED, EV_BOMB_DROPPED,
)

N_CHANNELS = 8  # walls, crates, coins, danger, bomb_timer, self, teammates(unused=0), opponents


def encode_obs(sim: VecBomberman, agent_slot: np.ndarray) -> np.ndarray:
    """
    agent_slot: (N,) int, which slot each env's observation is "from the
    perspective of" (self-centric channel split).
    Returns (N, N_CHANNELS, C, R) float32, values roughly in [-1, 1]/[0,1].
    """
    N, C, R = sim.n, sim.C, sim.R
    rows = np.arange(N)
    obs = np.zeros((N, N_CHANNELS, C, R), dtype=np.float32)
    obs[:, 0] = (sim.arena == WALL)
    obs[:, 1] = (sim.arena == CRATE)

    coin_grid = np.zeros((N, C, R), dtype=np.float32)
    collectable = sim.coin_state == 1
    for k in range(sim.K):
        sel = collectable[:, k]
        if sel.any():
            coin_grid[sel, sim.coin_xy[sel, k, 0], sim.coin_xy[sel, k, 1]] = 1.0
    obs[:, 2] = coin_grid

    obs[:, 3] = sim.danger_map().astype(np.float32) / max(sim.s.explosion_timer, 1)

    bomb_grid = np.zeros((N, C, R), dtype=np.float32)
    for slot in range(4):
        active = sim.bomb_active[:, slot]
        if active.any():
            bomb_grid[active, sim.bomb_xy[active, slot, 0], sim.bomb_xy[active, slot, 1]] = \
                sim.bomb_timer[active, slot].astype(np.float32) / max(sim.s.bomb_timer, 1)
    obs[:, 4] = bomb_grid

    self_grid = np.zeros((N, C, R), dtype=np.float32)
    alive_self = sim.agent_alive[rows, agent_slot]
    self_grid[alive_self, sim.agent_xy[rows, agent_slot, 0][alive_self],
              sim.agent_xy[rows, agent_slot, 1][alive_self]] = 1.0
    obs[:, 5] = self_grid

    opp_grid = np.zeros((N, C, R), dtype=np.float32)
    for slot in range(4):
        is_other = (slot != agent_slot) & sim.agent_alive[:, slot]
        if is_other.any():
            opp_grid[is_other, sim.agent_xy[is_other, slot, 0], sim.agent_xy[is_other, slot, 1]] = 1.0
    obs[:, 6] = opp_grid

    obs[:, 7] = sim.step_no[:, None, None].astype(np.float32) / sim.s.max_steps
    return obs


def default_reward(events: np.ndarray) -> np.ndarray:
    """events: (..., ) uint16 bit flags -> shaped reward, same shape."""
    r = np.zeros(events.shape, dtype=np.float32)
    r += 1.0 * ((events & EV_COIN_COLLECTED) != 0)
    r += 5.0 * ((events & EV_KILLED_OPPONENT) != 0)
    r -= 5.0 * ((events & EV_KILLED_SELF) != 0)
    r -= 5.0 * ((events & EV_GOT_KILLED) != 0)
    r -= 0.01 * ((events & EV_INVALID) != 0)
    r += 0.1 * ((events & EV_CRATE_DESTROYED) != 0)
    return r


class RandomOpponent:
    """Fills in actions for non-learning slots. Swap in a trained policy
    (or a rule-based bot from bots.py) for stronger training partners /
    self-play."""

    def __call__(self, sim: VecBomberman, slots: np.ndarray) -> np.ndarray:
        # slots: (N, k) which slots need actions filled in
        return np.random.randint(0, 6, size=slots.shape)


class BombermanEnv(gym.Env):
    """Single-game Gymnasium Env. Learning agent is always slot 0."""

    metadata = {"render_modes": []}

    def __init__(self, settings: Settings | None = None, opponent=None, seed: int = 0):
        super().__init__()
        self.sim = VecBomberman(1, settings, seed=seed)
        self.opponent = opponent or RandomOpponent()
        self.action_space = spaces.Discrete(6)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(N_CHANNELS, self.sim.C, self.sim.R), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.sim.reset(np.array([0]))
        obs = encode_obs(self.sim, np.array([0]))[0]
        return obs, {}

    def step(self, action: int):
        actions = np.zeros((1, 4), dtype=np.int64)
        actions[0, 0] = action
        other_slots = np.array([[1, 2, 3]])
        actions[0, 1:4] = self.opponent(self.sim, other_slots)[0]
        events, done = self.sim.step(actions)
        obs = encode_obs(self.sim, np.array([0]))[0]
        reward = float(default_reward(events[0, 0]))
        terminated = bool(done[0])
        truncated = bool(self.sim.step_no[0] >= self.sim.s.max_steps and terminated)
        info = {"events": int(events[0, 0]), "score": int(self.sim.agent_score[0, 0])}
        return obs, reward, terminated, truncated, info


class BombermanVecEnv:
    """
    stable-baselines3 VecEnv-compatible wrapper. This is the high-throughput
    path: `num_envs` games are advanced in a single sim.step() call.
    Learning agent is slot 0 in every game; slots 1-3 use `opponent`.
    """

    def __init__(self, num_envs: int, settings: Settings | None = None, opponent=None, seed: int = 0):
        self.num_envs = num_envs
        self.sim = VecBomberman(num_envs, settings, seed=seed)
        self.opponent = opponent or RandomOpponent()
        self.action_space = spaces.Discrete(6)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(N_CHANNELS, self.sim.C, self.sim.R), dtype=np.float32)
        self._actions = None
        self.render_mode = None

    def reset(self):
        self.sim.reset_all()
        return encode_obs(self.sim, np.zeros(self.num_envs, dtype=np.int64))

    def step_async(self, actions):
        self._actions = np.asarray(actions)

    def step_wait(self):
        full_actions = np.zeros((self.num_envs, 4), dtype=np.int64)
        full_actions[:, 0] = self._actions
        other_slots = np.tile(np.array([1, 2, 3]), (self.num_envs, 1))
        full_actions[:, 1:4] = self.opponent(self.sim, other_slots)

        events, done = self.sim.step(full_actions)
        rewards = default_reward(events[:, 0])
        infos = [{"score": int(self.sim.agent_score[n, 0])} for n in range(self.num_envs)]
        for n in np.nonzero(done)[0]:
            infos[n]["terminal_observation"] = None  # filled below if needed
            # "won": slot 0 outlasted everyone, or (on a timeout with >1
            # survivor) finished with the single highest score. Computed
            # here, before auto_reset_done() below overwrites this episode's
            # state -- lets a training callback track win-rate vs opponents.
            n_alive = int(self.sim.agent_alive[n].sum())
            if n_alive <= 1:
                infos[n]["won"] = bool(self.sim.agent_alive[n, 0])
            else:
                scores = self.sim.agent_score[n]
                infos[n]["won"] = bool(scores[0] == scores.max() and scores[0] > 0)

        self.sim.auto_reset_done(done)
        obs = encode_obs(self.sim, np.zeros(self.num_envs, dtype=np.int64))
        return obs, rewards, done, infos

    def step(self, actions):
        self.step_async(actions)
        return self.step_wait()

    def close(self):
        pass

    def get_attr(self, attr_name, indices=None):
        return [getattr(self, attr_name)] * self.num_envs

    def set_attr(self, attr_name, value, indices=None):
        setattr(self, attr_name, value)

    def env_method(self, method_name, *args, indices=None, **kwargs):
        raise NotImplementedError

    def env_is_wrapped(self, wrapper_class, indices=None):
        return [False] * self.num_envs

    def seed(self, seed=None):
        return [seed] * self.num_envs

## 5. Baseline: environment-only throughput (`benchmark.py`)

This measures the NumPy sim alone -- no torch involved. Run it first to confirm the environment itself is fast
(it should be, regardless of CPU vs GPU, since it's pure NumPy).

In [ ]:
%%writefile benchmark.py
"""
python benchmark.py

Reports two numbers per batch size:
  - "physics only": sim.step() alone (no obs encoding, no opponent policy,
    no resets) -- the ceiling for this NumPy implementation.
  - "full pipeline": step + observation encoding + opponent policy + auto
    reset, i.e. what you'd actually see driving PPO/etc through
    gym_env.BombermanVecEnv.
"""
import time
import numpy as np

from sim import VecBomberman, Settings
from gym_env import encode_obs
from bots import RandomSafeBot


def bench_physics_only(n_envs, n_steps=200, seed=0):
    sim = VecBomberman(n_envs, Settings(scenario="classic"), seed=seed)
    rng = np.random.default_rng(seed)
    t0 = time.time()
    for _ in range(n_steps):
        sim.step(rng.integers(0, 6, size=(n_envs, 4)))
    return n_envs * n_steps / (time.time() - t0)


def bench_full_pipeline(n_envs, n_steps=200, seed=0):
    sim = VecBomberman(n_envs, Settings(scenario="classic"), seed=seed)
    bot = RandomSafeBot()
    rng = np.random.default_rng(seed)
    slots = np.tile(np.array([1, 2, 3]), (n_envs, 1))
    t0 = time.time()
    for _ in range(n_steps):
        actions = np.zeros((n_envs, 4), dtype=np.int64)
        actions[:, 0] = rng.integers(0, 6, size=n_envs)
        actions[:, 1:4] = bot(sim, slots)
        events, done = sim.step(actions)
        _ = encode_obs(sim, np.zeros(n_envs, dtype=np.int64))
        sim.auto_reset_done(done)
    return n_envs * n_steps / (time.time() - t0)


if __name__ == "__main__":
    print(f"{'n_envs':>8} | {'physics only (steps/s)':>24} | {'full pipeline (steps/s)':>24}")
    for n_envs in [1, 64, 512, 2048, 8192]:
        p = bench_physics_only(n_envs)
        f = bench_full_pipeline(n_envs)
        print(f"{n_envs:>8} | {p:>24,.0f} | {f:>24,.0f}")


In [ ]:
!python benchmark.py

## 6. Isolate torch/GPU compute cost (`isolate_torch_costs.py`)

Same tiny CNN as the PPO feature extractor, run with no gym/SB3/sim involved at all. On CPU this was the
bottleneck (~900 samples/s); on a Colab GPU this should jump by 1-2 orders of magnitude.

In [ ]:
%%writefile isolate_torch_costs.py
"""
Pure-torch benchmark, no gym/SB3/sim involved at all. Isolates whether
slow training is a torch/CPU-environment problem (this script is also
slow) or specific to the PPO/rollout-buffer path (this script is fast,
so the problem is elsewhere -- e.g. memory/swap pressure from the
rollout buffer).

    python isolate_torch_costs.py

While this runs, in another terminal watch memory pressure:

    free -h -s 2       # updates every 2s; watch "available" and "swap"
    # or: htop / vmstat 2

--- Colab/GPU modification ---
Model and data are now moved to `device` (cuda if available, else cpu),
and we `torch.cuda.synchronize()` before/after timing -- CUDA ops are
launched asynchronously, so without a sync the wall-clock would only
measure how fast Python can *queue* kernels, not how long they actually
take to run.
"""
import time
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(f"torch.get_num_threads() = {torch.get_num_threads()}")
print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"torch.cuda.get_device_name(0) = {torch.cuda.get_device_name(0)}")


class SmallGridCNN(nn.Module):
    def __init__(self, features_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.linear = nn.Sequential(nn.Linear(64 * 17 * 17, features_dim), nn.ReLU())
        self.head = nn.Linear(features_dim, 6)  # stand-in policy head

    def forward(self, x):
        return self.head(self.linear(self.cnn(x)))


model = SmallGridCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

batch_size = 4096
n_minibatches = 16
n_epochs = 10
total_passes = n_minibatches * n_epochs

print(f"Running {total_passes} forward+backward passes of batch_size={batch_size} "
      f"on shape (8,17,17) -- matching one PPO train() call's workload...")

if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()
for _ in range(total_passes):
    x = torch.rand(batch_size, 8, 17, 17, device=device)
    target = torch.randint(0, 6, (batch_size,), device=device)
    opt.zero_grad()
    out = model(x)
    loss = nn.functional.cross_entropy(out, target)
    loss.backward()
    opt.step()
if torch.cuda.is_available():
    torch.cuda.synchronize()
dt = time.time() - t0

print(f"Total: {dt:.2f}s for {total_passes} passes "
      f"({total_passes * batch_size / dt:,.0f} samples/s)")
print()
print("If this number is also very slow (tens/hundreds of seconds), the "
      "problem is torch/CPU-environment-level, not our training code.")
print("If this is fast (a few seconds or less) but the real model.train() "
      "call is still slow, the difference is memory/swap pressure from "
      "the much larger rollout buffer -- check `free -h` while training, "
      "or (on GPU) whether the bottleneck moved to CPU<->GPU data transfer.")


In [ ]:
!python isolate_torch_costs.py

## 7. Self-play PPO training script (`train_example.py`)

GPU modifications: `device="cuda"` on the `PPO(...)` call, and `SmallGridCNN` widened
(64/128/128 channels, `features_dim=512`) since compute is no longer the constraint it was on CPU.

In [ ]:
%%writefile train_example.py
"""
Example: train PPO via self-play, with TensorBoard logging.

    pip install stable-baselines3 tensorboard
    python train_example.py
    tensorboard --logdir runs/          # then open http://localhost:6006

Self-play
---------
Opponent slots (1-3) are driven by bots.SelfPlayOpponent: a rotating pool
of frozen snapshots of the learner's own policy (bots.FrozenPolicy). A
callback below adds a new snapshot to the pool every `snapshot_freq`
timesteps and drops the oldest once the pool exceeds `pool_size`, so the
learner keeps facing a mix of recent and older versions of itself instead
of only its current (strongest) self -- the usual guard against self-play
strategy cycling / collapse. Before the first snapshot exists, opponents
just act randomly (bots.RandomBot), so training starts immediately.

TensorBoard
-----------
- VecMonitor wraps the vec env so SB3 logs the standard rollout/ep_rew_mean
  and rollout/ep_len_mean scalars automatically.
- SelfPlayCallback additionally logs:
    selfplay/win_rate_vs_pool  -- rolling win rate over the last 200
                                  finished episodes (see gym_env.py's
                                  "won" info flag)
    selfplay/pool_size         -- how many frozen snapshots are active
- PPO's own training diagnostics (loss, KL, clip fraction, explained
  variance, fps, etc.) are logged under train/ as usual.

Once this is working, you can drop in bots.RandomSafeBot (or a mix, via
your own opponent callable) instead of/alongside SelfPlayOpponent if you
want a scripted baseline to periodically re-check against.

--- Colab/GPU modifications ---
1. `device="cuda"` is passed to PPO(...) so the policy (including the
   SmallGridCNN feature extractor below) and its optimizer live on the
   GPU. bots.FrozenPolicy deep-copies model.policy, so self-play
   opponent snapshots inherit the same device automatically -- no
   change needed in bots.py.
2. On CPU, the network's compute cost was the bottleneck, so it had to
   stay small. On GPU that constraint mostly disappears: channel counts
   are widened (32/64/64 -> 64/128/128) and features_dim is doubled
   (256 -> 512), since a GPU parallelizes this batched conv+dense work
   far more cheaply than a CPU does.
"""
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.vec_env.base_vec_env import VecEnv as SB3VecEnv

from gym_env import BombermanVecEnv
from bots import SelfPlayOpponent, FrozenPolicy
from sim import Settings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class SmallGridCNN(BaseFeaturesExtractor):
    """SB3's default CnnPolicy feature extractor (NatureCNN) assumes
    ~84x84 Atari-style frames -- its first conv layer alone (kernel 8,
    stride 4) needs an input bigger than our 17x17 board, which is why
    plugging CnnPolicy in directly fails with a "kernel size can't be
    greater than input size" error. This is a shallow stride-1 CNN sized
    for a small grid instead: spatial size is preserved through the conv
    stack (padding=1) rather than aggressively downsampled.

    Widened for GPU (64/128/128 channels, features_dim=512 by default)
    now that compute is cheap -- on CPU this same width would have made
    training slower, not faster."""

    def __init__(self, observation_space, features_dim: int = 512):
        super().__init__(observation_space, features_dim)
        n_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            sample = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))


class SB3Compatible(BombermanVecEnv, SB3VecEnv):
    """BombermanVecEnv already implements the VecEnv method surface;
    this just satisfies SB3's isinstance check."""

    def __init__(self, num_envs, settings=None, opponent=None, seed=0):
        BombermanVecEnv.__init__(self, num_envs, settings, opponent, seed)


class SelfPlayCallback(BaseCallback):
    """Periodically freezes the current policy into the opponent's
    snapshot pool and logs self-play metrics to TensorBoard."""

    def __init__(self, opponent: SelfPlayOpponent, snapshot_freq: int, verbose: int = 0):
        super().__init__(verbose)
        self.opponent = opponent
        self.snapshot_freq = snapshot_freq
        self._recent_wins: list[float] = []

    def _on_step(self) -> bool:
        for info, done in zip(self.locals.get("infos", []), self.locals.get("dones", [])):
            if done and "won" in info:
                self._recent_wins.append(float(info["won"]))
        self._recent_wins = self._recent_wins[-200:]

        if self._recent_wins:
            self.logger.record("selfplay/win_rate_vs_pool", float(np.mean(self._recent_wins)))
        self.logger.record("selfplay/pool_size", len(self.opponent.policies))

        if self.n_calls % self.snapshot_freq == 0:
            self.opponent.add_snapshot(self.model.policy)
            if self.verbose:
                print(f"[selfplay] snapshot #{len(self.opponent.policies)} "
                      f"added at step {self.num_timesteps}")
        return True


def main():
    print(f"Training on device: {DEVICE}")
    n_envs = 2048
    opponent = SelfPlayOpponent(pool_size=5)
    venv = SB3Compatible(n_envs, Settings(scenario="classic"), opponent=opponent)
    venv = VecMonitor(venv)  # -> rollout/ep_rew_mean, rollout/ep_len_mean on TensorBoard

    model = PPO(
        "CnnPolicy",
        venv,
        device=DEVICE,        # <-- GPU: policy + optimizer live on cuda
        n_steps=512,          # per env -> 128*512 = 65536 transitions/update
        batch_size=16*1024,
        learning_rate=3e-4,
        n_epochs=10,
        verbose=1,
        tensorboard_log="./runs",
        policy_kwargs=dict(
            features_extractor_class=SmallGridCNN,
            features_extractor_kwargs=dict(features_dim=512),
            normalize_images=False,
        ),
    )

    callback = SelfPlayCallback(opponent, snapshot_freq=50_000, verbose=1)
    model.learn(total_timesteps=2_000_000, callback=callback, tb_log_name="ppo_selfplay")
    model.save("bomberman_ppo")


if __name__ == "__main__":
    main()


## 8. Profile where the time goes (`profile_train.py`)

Times, separately: (1) physics only, (2) + observation encoding + self-play opponent inference,
(3) PPO's `model.train()` gradient step -- now all torch-side work runs on GPU (step 1 stays on CPU/NumPy,
since the sim has no GPU path and was never the bottleneck).

In [ ]:
%%writefile profile_train.py
"""
Run this to see where the fps is actually going:

    python profile_train.py

It times, separately:
  1. sim.step() alone                      (physics ceiling, no torch)
  2. + encode_obs + SelfPlayOpponent        (rollout collection cost)
  3. PPO's model.train() gradient step      (backward-pass cost)

Compare (2) against benchmark.py's "full pipeline" number for the same
num_envs to see how much the self-play opponent specifically costs vs.
the RandomSafeBot baseline it replaced. Compare (2) vs (3) to see whether
rollout collection or gradient updates dominate total wall-clock.

--- Colab/GPU modification ---
`device=DEVICE` is passed to PPO(...) so step 3 (model.train()) runs on
GPU. Step 2 (self-play opponent inference via bots.FrozenPolicy) also
moves to GPU automatically, since FrozenPolicy deep-copies model.policy.
sim.step() itself (step 1) stays on CPU/NumPy either way -- vectorized
NumPy has no GPU path here, and at ~150k steps/s it was never the
bottleneck.
"""
import time
import numpy as np
import torch

from sim import Settings
from gym_env import encode_obs
from bots import SelfPlayOpponent
from train_example import SmallGridCNN, SB3Compatible, DEVICE
from stable_baselines3 import PPO

N_ENVS = 512
N_STEPS = 100


def main():
    print(f"device = {DEVICE}")
    print(f"torch.get_num_threads() = {torch.get_num_threads()}")
    print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")

    opponent = SelfPlayOpponent(pool_size=5)
    venv = SB3Compatible(N_ENVS, Settings(scenario="classic"), opponent=opponent)
    model = PPO(
        "CnnPolicy", venv, device=DEVICE, n_steps=128, batch_size=4096, verbose=0,
        policy_kwargs=dict(features_extractor_class=SmallGridCNN,
                            features_extractor_kwargs=dict(features_dim=512),
                            normalize_images=False),
    )
    for _ in range(5):
        opponent.add_snapshot(model.policy)  # simulate a "warmed up" pool

    sim = venv.sim
    slots = np.tile(np.array([1, 2, 3]), (N_ENVS, 1))
    rng = np.random.default_rng(0)

    # 1. physics only
    t0 = time.time()
    for _ in range(N_STEPS):
        sim.step(rng.integers(0, 6, size=(N_ENVS, 4)))
    dt = time.time() - t0
    print(f"1. physics only:            {N_ENVS*N_STEPS/dt:>12,.0f} steps/s  ({dt/N_STEPS*1000:.2f} ms/step)")

    # 2. + encode_obs + self-play opponent (the actual rollout-collection cost)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(N_STEPS):
        actions = np.zeros((N_ENVS, 4), dtype=np.int64)
        actions[:, 0] = rng.integers(0, 6, size=N_ENVS)
        actions[:, 1:4] = opponent(sim, slots)
        events, done = sim.step(actions)
        _ = encode_obs(sim, np.zeros(N_ENVS, dtype=np.int64))
        sim.auto_reset_done(done)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.time() - t0
    print(f"2. + obs + selfplay opp:    {N_ENVS*N_STEPS/dt:>12,.0f} steps/s  ({dt/N_STEPS*1000:.2f} ms/step)")

    # 3. PPO gradient update cost (collect one real rollout, then time .train())
    model.learn(total_timesteps=N_ENVS * 128)  # fills the rollout buffer once
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    model.train()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.time() - t0
    rollout_size = N_ENVS * 128
    print(f"3. model.train() on {rollout_size} samples: {dt:.2f}s  "
          f"({rollout_size/dt:,.0f} samples/s equivalent)")


if __name__ == "__main__":
    main()


In [ ]:
!python profile_train.py

## 9. Full training run

Launches the actual self-play PPO run (2,000,000 timesteps by default -- edit `total_timesteps` in
`train_example.py`'s `main()` above, or override below, to fit your Colab session length).

TensorBoard logs go to `./runs`; view them live with the `%tensorboard` cell after this starts, or in a
separate cell while training runs in the background.

In [ ]:
import train_example
train_example.main()


### (Optional) Save a checkpoint to Drive

If you mounted Drive in step 3, copy the saved model there so it survives a disconnect.

In [ ]:
import shutil, os
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy("bomberman_ppo.zip", os.path.join(CHECKPOINT_DIR, "bomberman_ppo.zip"))
    print("Saved to", CHECKPOINT_DIR)
else:
    print("Drive not mounted -- skipping. Model is still available locally as bomberman_ppo.zip")


## 10. (Optional) View TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/
